### C. Abscissum genome: 

https://www.ncbi.nlm.nih.gov/datasets/genome/GCA_023376855.1/

In [24]:
c_abs_CDS_file = 'c_abscissum_data/ncbi_dataset/data/GCA_023376855.1/cds_from_genomic.fna'

In [25]:
# Função de Inspeção de Arquivos FASTA (Quality Control)
import gzip
from Bio import SeqIO

def inspecionar_genoma(caminho_arquivo):
    """
    Lê um arquivo FASTA (compactado ou não) e gera um relatório estatístico.
    Útil para verificar se baixamos o arquivo correto (CDS vs Genoma completo).
    """
    print(f"--- RELATÓRIO DE INSPEÇÃO: {caminho_arquivo} ---")
    
    # Detecção automática de gzip
    if caminho_arquivo.endswith(".gz"):
        handle = gzip.open(caminho_arquivo, "rt") # rt = read text
    else:
        handle = open(caminho_arquivo, "r")

    # Variáveis para estatísticas
    total_sequencias = 0
    total_bases = 0
    comprimentos = []
    tipos_ids = []
    
    try:
        # Itera pelo arquivo apenas uma vez para não lotar a memória RAM
        for record in SeqIO.parse(handle, "fasta"):
            seq_len = len(record.seq)
            total_sequencias += 1
            total_bases += seq_len
            comprimentos.append(seq_len)
            
            # Guarda os primeiros 3 IDs para amostragem
            if total_sequencias <= 3:
                tipos_ids.append((record.id, record.description))
                
        # Fecha o arquivo
        handle.close()
        
        if total_sequencias == 0:
            print("[ERRO] O arquivo está vazio ou não é um FASTA válido.")
            return

        # Cálculos Finais
        avg_len = total_bases / total_sequencias
        min_len = min(comprimentos)
        max_len = max(comprimentos)
        
        # --- IMPRESSÃO DO RELATÓRIO ---
        print(f"1. Quantidade Total de Sequências: {total_sequencias}")
        print("-" * 40)
        
        print(f"2. Tamanho (Pares de Base):")
        print(f"   - Mínimo: {min_len} pb")
        print(f"   - Máximo: {max_len} pb")
        print(f"   - Média:  {avg_len:.2f} pb")
        print("-" * 40)
        
        print(f"3. Amostra de Cabeçalhos (IDs):")
        for i, (rid, desc) in enumerate(tipos_ids):
            print(f"   Seq {i+1}: {rid}")
            print(f"          Desc: {desc[:80]}...") # Corta descrições muito longas
        
        print("=" * 60 + "\n")

    except FileNotFoundError:
        print(f"[ERRO CRÍTICO] Arquivo não encontrado no caminho: {caminho_arquivo}")
    except Exception as e:
        print(f"[ERRO] Falha ao ler o arquivo: {e}")

In [26]:
inspecionar_genoma(c_abs_CDS_file)

--- RELATÓRIO DE INSPEÇÃO: c_abscissum_data/ncbi_dataset/data/GCA_023376855.1/cds_from_genomic.fna ---
1. Quantidade Total de Sequências: 15499
----------------------------------------
2. Tamanho (Pares de Base):
   - Mínimo: 54 pb
   - Máximo: 28416 pb
   - Média:  1392.71 pb
----------------------------------------
3. Amostra de Cabeçalhos (IDs):
   Seq 1: lcl|SDAQ01000001.1_cds_KAI3559026.1_1
          Desc: lcl|SDAQ01000001.1_cds_KAI3559026.1_1 [locus_tag=CABS02_00001] [protein=hypothet...
   Seq 2: lcl|SDAQ01000001.1_cds_KAI3559027.1_2
          Desc: lcl|SDAQ01000001.1_cds_KAI3559027.1_2 [locus_tag=CABS02_00002] [protein=serine/t...
   Seq 3: lcl|SDAQ01000001.1_cds_KAI3559028.1_3
          Desc: lcl|SDAQ01000001.1_cds_KAI3559028.1_3 [locus_tag=CABS02_00003] [protein=alpha/be...



In [27]:
# Busca Inteligente por Alvos de Patogenicidade

# Lista de termos chave que indicam genes essenciais para a sobrevivência e patogenicidade do fungo. Esses termos são comuns em descrições de genes que codificam proteínas envolvidas em processos críticos como síntese de melanina, formação da parede celular, divisão celular, e produção de efetores.
termos_de_interesse = [
    "polyketide synthase",   # Para melanina (PKS1)
    "chitin synthase",       # Para parede celular (CHS)
    "septin",                # Importante para divisão celular
    "effector",              # Proteínas de ataque
    "scytalone dehydratase"  # Também da via da melanina
]

print(f"--- Iniciando busca por {len(termos_de_interesse)} termos estratégicos ---")

genes_encontrados = {} # Dicionário para guardar os resultados

# Varre todo o genoma (pode levar alguns segundos)
c_abs_genes = SeqIO.index(c_abs_CDS_file, "fasta")
for record_id in c_abs_genes:
    registro = c_abs_genes[record_id]
    descricao_lower = registro.description.lower()
    
    for termo in termos_de_interesse:
        if termo in descricao_lower:
            # Se achou, salva na lista desse termo
            if termo not in genes_encontrados:
                genes_encontrados[termo] = []
            genes_encontrados[termo].append(registro)

# Exibe os resultados
if not genes_encontrados:
    print("Nenhum termo encontrado. Tente palavras mais genéricas (ex: 'kinase', 'synthase').")
else:
    for termo, lista in genes_encontrados.items():
        print(f"\n>>> TERMO: '{termo.upper()}' - Encontrados: {len(lista)}")
        # Mostra o primeiro de cada grupo como exemplo
        primeiro = lista[0]
        print(f"   Exemplo de ID: {primeiro.id}")
        print(f"   Descrição: {primeiro.description}")
        print(f"   Tamanho: {len(primeiro.seq)} pb")

--- Iniciando busca por 5 termos estratégicos ---

>>> TERMO: 'POLYKETIDE SYNTHASE' - Encontrados: 5
   Exemplo de ID: lcl|SDAQ01000011.1_cds_KAI3556698.1_3138
   Descrição: lcl|SDAQ01000011.1_cds_KAI3556698.1_3138 [locus_tag=CABS02_03139] [protein=polyketide synthase] [protein_id=KAI3556698.1] [location=complement(join(478089..479482,479929..480036,480084..480362,480636..485228,485286..485735,485800..486285,486513..486636,486685..487204,487266..487771))] [gbkey=CDS]
   Tamanho: 8460 pb

>>> TERMO: 'SCYTALONE DEHYDRATASE' - Encontrados: 3
   Exemplo de ID: lcl|SDAQ01000016.1_cds_KAI3555617.1_3992
   Descrição: lcl|SDAQ01000016.1_cds_KAI3555617.1_3992 [locus_tag=CABS02_03993] [protein=scytalone dehydratase] [protein_id=KAI3555617.1] [location=complement(join(36521..36964,37036..37133,37189..37222))] [gbkey=CDS]
   Tamanho: 576 pb

>>> TERMO: 'CHITIN SYNTHASE' - Encontrados: 9
   Exemplo de ID: lcl|SDAQ01000025.1_cds_KAI3554229.1_5643
   Descrição: lcl|SDAQ01000025.1_cds_KAI3554229.1_564

#### Vamos focar na Scytalone Dehydratase (SCD) (o segundo resultado da lista).

Por que este gene? Ele é pequeno (576 pb), o que facilita a visualização, e é crítico para a síntese de melanina. Se o produto funcionar, o fungo ficará "albino" e incapaz de infectar a planta (o apressório falha).

In [28]:
# --- 1. CONFIGURAÇÃO DO ALVO ---
# ID da Scytalone Dehydratase encontrado na busca anterior
id_gene_alvo = "lcl|SDAQ01000016.1_cds_KAI3555617.1_3992" 
tamanho_kmer = 21  # Tamanho padrão do siRNA (Dicer)

In [29]:
# Simulador de Especificidade (Scanner de K-mers)

# CARREGAR (OU SIMULAR) O BANCO DE "OFF-TARGETS" ---
# Em um cenário real, leríamos o arquivo do Citros aqui.
# Para este teste, vamos criar um conjunto fictício de sequências "perigosas"
# para ver se o script detecta o risco.
print("Simulando banco de sequências proibidas (Genoma do Hospedeiro)...")
kmers_proibidos = set()

# Vamos "contaminar" propositalmente o banco com um pedaço do próprio gene
# só para testar se o script detecta o perigo.
if id_gene_alvo in c_abs_genes:
    seq_completa = str(c_abs_genes[id_gene_alvo].seq).upper()
    # Pegamos um trecho do meio do gene (bases 200 a 250) e marcamos como "proibido"
    trecho_perigoso = seq_completa[200:250]
    
    # Adiciona os k-mers desse trecho ao banco de proibidos
    for i in range(len(trecho_perigoso) - tamanho_kmer + 1):
        kmers_proibidos.add(trecho_perigoso[i:i+tamanho_kmer])
    
    print(f"Simulação: Adicionados {len(kmers_proibidos)} k-mers proibidos ao banco.")
else:
    print("ERRO: ID do gene não encontrado no índice.")

# --- 3. A VARREDURA (SCANNER) ---
print(f"\n--- Iniciando Varredura no Gene: {id_gene_alvo} ---")

if id_gene_alvo in c_abs_genes:
    seq_gene = str(c_abs_genes[id_gene_alvo].seq).upper()
    mapa_visual = ""
    regioes_seguras = []
    inicio_seguro_atual = None
    
    # Loop da Janela Deslizante (Sliding Window)
    for i in range(len(seq_gene) - tamanho_kmer + 1):
        kmer_atual = seq_gene[i : i+tamanho_kmer]
        
        if kmer_atual in kmers_proibidos:
            # PERIGO: Esse pedaço existe no hospedeiro!
            mapa_visual += "X" 
            if inicio_seguro_atual is not None:
                # Fecha a região segura anterior
                regioes_seguras.append((inicio_seguro_atual, i))
                inicio_seguro_atual = None
        else:
            # SEGURO: Esse pedaço é exclusivo do fungo
            mapa_visual += "."
            if inicio_seguro_atual is None:
                inicio_seguro_atual = i
    
    # Fecha a última região se existir
    if inicio_seguro_atual is not None:
        regioes_seguras.append((inicio_seguro_atual, len(seq_gene)))

    # --- 4. RELATÓRIO FINAL ---
    print("\n[MAPA DE RISCO]")
    print("Legenda: (.) = Seguro | (X) = Risco de Silenciar o Hospedeiro")
    print(f"Sequência: {mapa_visual[:100]}... (mostrando primeiros 100pb)")
    
    print("\n[RESULTADO DO DESIGN DE dsRNA]")
    print(f"Tamanho total do gene: {len(seq_gene)} pb")
    
    encontrou_bom_alvo = False
    print("\nRegiões Recomendadas para Síntese:")
    for inicio, fim in regioes_seguras:
        tamanho = fim - inicio
        # Só nos interessa fragmentos maiores que 100pb para fabricar o dsRNA
        if tamanho > 100:
            encontrou_bom_alvo = True
            print(f"  >>> DE {inicio} A {fim} (Tamanho: {tamanho} pb) - EXCELENTE")
            # Extrair a sequência para enviar para síntese
            seq_para_sintese = seq_gene[inicio:fim]
            print(f"      Sequência: {seq_para_sintese[:30]}...[continua]")
        elif tamanho > 50:
            print(f"  > De {inicio} a {fim} ({tamanho} pb) - Razoável")
            
    if not encontrou_bom_alvo:
        print("\nALERTA: Nenhuma região contínua >100pb foi encontrada. Tente outro gene.")

else:
    print("Gene não encontrado. Verifique o ID.")

Simulando banco de sequências proibidas (Genoma do Hospedeiro)...
Simulação: Adicionados 30 k-mers proibidos ao banco.

--- Iniciando Varredura no Gene: lcl|SDAQ01000016.1_cds_KAI3555617.1_3992 ---

[MAPA DE RISCO]
Legenda: (.) = Seguro | (X) = Risco de Silenciar o Hospedeiro
Sequência: ....................................................................................................... (mostrando primeiros 100pb)

[RESULTADO DO DESIGN DE dsRNA]
Tamanho total do gene: 576 pb

Regiões Recomendadas para Síntese:
  >>> DE 0 A 200 (Tamanho: 200 pb) - EXCELENTE
      Sequência: ATGGCGAGCCCTGCTGGAAACATCTCCTTT...[continua]
  >>> DE 230 A 576 (Tamanho: 346 pb) - EXCELENTE
      Sequência: CCTCAAGACCCAGCACTTTATCGGCGGCAC...[continua]


O script funcionou como planejado.

Vamos interpretar o que aconteceu:

O "Buraco" (200 a 230): Note que o script recomendou a região 0-200 e depois pulou para 230-576.

Por que? Porque na simulação nós inserimos propositalmente um trecho "perigoso" entre as bases 200 e 250.

Conclusão: O algoritmo detectou a ameaça (match com o hospedeiro) e cortou essa parte automaticamente. Isso garante que o produto final não silencie a planta.

As Regiões "EXCELENTE": O script te deu dois fragmentos longos (200pb e 346pb).

Aplicação: Qualquer um desses dois pedaços poderia ser sintetizado como dsRNA.

### Citrus sinensis genome:

https://www.ncbi.nlm.nih.gov/datasets/genome/GCF_022201045.2/

In [32]:
c_sin_CDS_file = 'data_targets/c_sinensis_data/ncbi_dataset/data/GCF_022201045.2/cds_from_genomic.fna'

In [33]:
inspecionar_genoma(c_sin_CDS_file)

--- RELATÓRIO DE INSPEÇÃO: data_targets/c_sinensis_data/ncbi_dataset/data/GCF_022201045.2/cds_from_genomic.fna ---
1. Quantidade Total de Sequências: 40427
----------------------------------------
2. Tamanho (Pares de Base):
   - Mínimo: 90 pb
   - Máximo: 16296 pb
   - Média:  1534.59 pb
----------------------------------------
3. Amostra de Cabeçalhos (IDs):
   Seq 1: lcl|NC_068556.1_cds_XP_024956459.2_1
          Desc: lcl|NC_068556.1_cds_XP_024956459.2_1 [gene=LOC102606664] [db_xref=GeneID:1026066...
   Seq 2: lcl|NC_068556.1_cds_XP_006483668.2_2
          Desc: lcl|NC_068556.1_cds_XP_006483668.2_2 [gene=LOC102606664] [db_xref=GeneID:1026066...
   Seq 3: lcl|NC_068556.1_cds_XP_006483669.2_3
          Desc: lcl|NC_068556.1_cds_XP_006483669.2_3 [gene=LOC102606664] [db_xref=GeneID:1026066...



In [34]:
# Carregamento do Banco de Exclusão Real (Citrus sinensis)

# --- CONFIGURAÇÃO ---
k = 21 # Tamanho do siRNA (padrão Dicer)

import time

print(f"--- Carregando 'Lista Negra' de k-mers de: {c_sin_CDS_file} ---")
print("Isso pode demorar um pouco...")
start_time = time.time()

blacklist_kmers = set()

try:
    # Lógica para abrir arquivo normal ou compactado
    if c_sin_CDS_file.endswith(".gz"):
        import gzip
        handle = gzip.open(c_sin_CDS_file, "rt")
    else:
        handle = open(c_sin_CDS_file, "r")
    count_seqs = 0
    for record in SeqIO.parse(handle, "fasta"):
        seq = str(record.seq).upper()
        # Gera todos os k-mers dessa sequência
        for i in range(len(seq) - k + 1):
            blacklist_kmers.add(seq[i:i+k])
        count_seqs += 1
    
    handle.close()
    
    end_time = time.time()
    print(f"\n[SUCESSO] Banco carregado em {end_time - start_time:.2f} segundos.")
    print(f"Total de Genes Processados: {count_seqs}")
    print(f"Total de k-mers Únicos Proibidos: {len(blacklist_kmers)}")

except FileNotFoundError:
    print(f"\n[ERRO] Não encontrei o arquivo: {c_sin_CDS_file}")
    print("Verifique se o nome está correto e se o arquivo está na mesma pasta do notebook.")

--- Carregando 'Lista Negra' de k-mers de: data_targets/c_sinensis_data/ncbi_dataset/data/GCF_022201045.2/cds_from_genomic.fna ---
Isso pode demorar um pouco...

[SUCESSO] Banco carregado em 21.34 segundos.
Total de Genes Processados: 40427
Total de k-mers Únicos Proibidos: 29741838


In [35]:
# Varredura de Especificidade (Cruzamento Fungo vs Planta)

def varredura_especificidade(id_gene_alvo, db_genes, blacklist_kmers):

    print(f"--- Analisando Especificidade do Gene: {id_gene_alvo} ---")
    if id_gene_alvo in db_genes:
        seq_gene = str(db_genes[id_gene_alvo].seq).upper()
        tamanho_total = len(seq_gene)
        
        # Variáveis da Janela Deslizante
        regioes_seguras = []
        inicio_seguro = None
        conflitos_encontrados = 0
        mapa_visual = [] # Vamos guardar caracteres para o mapa

        # Varredura
        for i in range(tamanho_total - 21 + 1):
            kmer = seq_gene[i : i+21]
            
            if kmer in blacklist_kmers:
                # PERIGO: Match com Citros
                conflitos_encontrados += 1
                mapa_visual.append("X")
                if inicio_seguro is not None:
                    regioes_seguras.append((inicio_seguro, i))
                    inicio_seguro = None
            else:
                # SEGURO
                mapa_visual.append(".")
                if inicio_seguro is None:
                    inicio_seguro = i
        
        # Fecha a última região
        if inicio_seguro is not None:
            regioes_seguras.append((inicio_seguro, tamanho_total))

        # --- RELATÓRIO ---
        print(f"\nResumo da Análise:")
        print(f"1. Tamanho do Gene: {tamanho_total} pb")
        print(f"2. Conflitos (Off-targets): {conflitos_encontrados} k-mers perigosos encontrados.")
        
        percentual_seguro = 100 - (conflitos_encontrados / (tamanho_total-20) * 100)
        print(f"3. Índice de Segurança Global: {percentual_seguro:.1f}%")
        
        print("\n[MAPA VISUAL SIMPLIFICADO]")
        # Mostra o mapa em linhas de 100 caracteres para facilitar leitura
        str_mapa = "".join(mapa_visual)
        for i in range(0, len(str_mapa), 100):
            print(f"{i:04d}: {str_mapa[i:i+100]}")

        print("\n" + "="*60)
        print("COORDENADAS PARA SÍNTESE DE dsRNA (Safe Zones)")
        print("="*60)
        
        boas_opcoes = False
        for inicio, fim in regioes_seguras:
            tamanho = fim - inicio
            # Critério: Só queremos fragmentos > 150bp para ser eficiente
            if tamanho >= 150:
                boas_opcoes = True
                print(f">>> ALVO PREMIUM: Bases {inicio} a {fim} (Tamanho: {tamanho} bp)")
                seq_final = seq_gene[inicio:fim]
                print(f"    Sequência:")
                print(f"    {seq_final}\n")
            elif tamanho >= 80:
                print(f"> Alvo Secundário: Bases {inicio} a {fim} ({tamanho} bp) - Curto, mas usável.")
        
        if not boas_opcoes:
            print("\n[ATENÇÃO] Não encontramos regiões longas (>150bp) contínuas.")
            print("Sugestão: Tente outro gene da lista de candidatos.")

    else:
        print(f"Gene '{id_gene_alvo}' não encontrado no índice do fungo.")

In [36]:
# --- ESCOLHA DO ALVO ---
id_gene_alvo = "lcl|SDAQ01000016.1_cds_KAI3555617.1_3992"  # Exemplo: Scytalone Dehydratase
varredura_especificidade(id_gene_alvo, c_abs_genes, blacklist_kmers)

--- Analisando Especificidade do Gene: lcl|SDAQ01000016.1_cds_KAI3555617.1_3992 ---

Resumo da Análise:
1. Tamanho do Gene: 576 pb
2. Conflitos (Off-targets): 0 k-mers perigosos encontrados.
3. Índice de Segurança Global: 100.0%

[MAPA VISUAL SIMPLIFICADO]
0000: ....................................................................................................
0100: ....................................................................................................
0200: ....................................................................................................
0300: ....................................................................................................
0400: ....................................................................................................
0500: ........................................................

COORDENADAS PARA SÍNTESE DE dsRNA (Safe Zones)
>>> ALVO PREMIUM: Bases 0 a 576 (Tamanho: 576 bp)
    Sequência:
    ATGGCGAGCCCTGCTGGAAACATCTCCT

### Teste com controle positivo

A actina é uma proteína do citoesqueleto essencial para a divisão celular e motilidade. Devido à sua importância vital, trechos da sua sequência de DNA costumam ser altamente conservados.

In [37]:
# Busca Rigorosa pelo Controle Positivo (Actina ou Tubulina)
print("--- Iniciando busca rigorosa por Controle Positivo ---")

# Vamos tentar dois alvos clássicos. Se não achar um, tenta o outro.
termos_busca = ["actin", "tubulin"]
encontrado = False

for record_id in c_abs_genes:
    registro = c_abs_genes[record_id]
    desc = registro.description.lower()
    
    # FILTRO DE QUALIDADE:
    # 1. Ignora se tiver "interacting" (o erro anterior)
    # 2. Ignora "like" (actin-like protein não é actina)
    # 3. Ignora "regulator" ou "binding" (queremos a proteína estrutural)
    termos_banidos = ["interacting", "like", "binding", "related", "regulator", "family"]
    
    if any(banido in desc for banido in termos_banidos):
        continue # Pula para o próximo gene
        
    # Verifica se é um dos nossos alvos
    for termo in termos_busca:
        # Garante que o termo existe e não é parte de outra palavra (ex: 'acting')
        # Verifica se está no descritivo de forma limpa
        if termo in desc:
            print(f"\n[ALVO ESTRUTURAL ENCONTRADO: {termo.upper()}]")
            print(f"ID: {record_id}")
            print(f"Descrição: {registro.description}")
            print(f"Tamanho: {len(registro.seq)} pb")
            
            # Vamos priorizar a Tubulina se achar, pois é muito conservada em Colletotrichum
            if "tubulin beta" in desc or "actin" in desc:
                    # Sugestão visual para você saber que é o bom
                print(">>> RECOMENDAÇÃO: Use este ID para o teste de validação.")
            
            encontrado = True
            # Removemos o break para ele mostrar todas as opções limpas
            # break 

if not encontrado:
    print("Nenhum alvo estrutural limpo encontrado. A anotação do genoma pode estar atípica.")

--- Iniciando busca rigorosa por Controle Positivo ---

[ALVO ESTRUTURAL ENCONTRADO: TUBULIN]
ID: lcl|SDAQ01000011.1_cds_KAI3556572.1_3012
Descrição: lcl|SDAQ01000011.1_cds_KAI3556572.1_3012 [locus_tag=CABS02_03013] [protein=tubulin-specific chaperone Rbl2] [protein_id=KAI3556572.1] [location=complement(join(21419..21616,21730..21909))] [gbkey=CDS]
Tamanho: 378 pb

[ALVO ESTRUTURAL ENCONTRADO: TUBULIN]
ID: lcl|SDAQ01000015.1_cds_KAI3555828.1_3881
Descrição: lcl|SDAQ01000015.1_cds_KAI3555828.1_3881 [locus_tag=CABS02_03882] [protein=tubulin alpha-B chain] [protein_id=KAI3555828.1] [location=join(206397..206421,206576..206616,206755..206781,206845..206920,206993..207523,207597..208020,208079..208307)] [gbkey=CDS]
Tamanho: 1353 pb

[ALVO ESTRUTURAL ENCONTRADO: ACTIN]
ID: lcl|SDAQ01000019.1_cds_KAI3555194.1_4641
Descrição: lcl|SDAQ01000019.1_cds_KAI3555194.1_4641 [locus_tag=CABS02_04642] [protein=actin] [partial=5'] [protein_id=KAI3555194.1] [location=join(<389264..389508,389564..390131,390

In [38]:
# --- ESCOLHA DO ALVO ---
id_gene_alvo = "lcl|SDAQ01000033.1_cds_KAI3553165.1_6722"  # É uma "Actina" pura (não parcial, não reguladora). Tem o tamanho clássico (~1.3kb). Se existe um gene que vai dar "match" com a planta, é este.

varredura_especificidade(id_gene_alvo, c_abs_genes, blacklist_kmers)

--- Analisando Especificidade do Gene: lcl|SDAQ01000033.1_cds_KAI3553165.1_6722 ---

Resumo da Análise:
1. Tamanho do Gene: 1347 pb
2. Conflitos (Off-targets): 0 k-mers perigosos encontrados.
3. Índice de Segurança Global: 100.0%

[MAPA VISUAL SIMPLIFICADO]
0000: ....................................................................................................
0100: ....................................................................................................
0200: ....................................................................................................
0300: ....................................................................................................
0400: ....................................................................................................
0500: ....................................................................................................
0600: ..............................................................................................

In [39]:
#DIAGNÓSTICO DE FALHA
print("--- INICIANDO DIAGNÓSTICO DO SISTEMA ---")

# 1. Checagem de Carregamento
if 'blacklist_kmers' not in globals():
    print("[ERRO CRÍTICO] A variável 'blacklist_kmers' NÃO EXISTE.")
    print("Ação: Rode a Célula 7 novamente.")
else:
    tamanho_banco = len(blacklist_kmers)
    print(f"1. Tamanho do Banco de Citros na memória: {tamanho_banco} k-mers")
    
    if tamanho_banco < 1000:
        print("   [ALERTA VERMELHO] O banco está praticamente vazio! O arquivo de Citros não foi lido corretamente.")
    else:
        print("   [OK] O banco parece ter dados.")
        
        # 2. Checagem de Formato (Case Sensitivity)
        amostra = list(blacklist_kmers)[0]
        print(f"2. Amostra de um k-mer do Citros: '{amostra}'")
        
        if amostra.islower():
            print("   [ERRO IDENTIFICADO] O banco de Citros está em minúsculas (acgt)!")
            print("   O script do fungo usa MAIÚSCULAS. Eles nunca vão dar match.")
        else:
            print("   [OK] O banco está em MAIÚSCULAS.")

# 3. Teste de Match Forçado (Prova Real)
# Vamos pegar um pedaço da sequência da Actina do fungo que você acabou de postar
# e verificar manualmente se ele está no banco.

# Sequência retirada do seu output (bases 100 a 121)
# ATGGCTGGTGGCCGCAATACCA (início da actina)
seq_teste = "ATGGCTGGTGGCCGCAATACC" # 21pb

print(f"\n3. Teste Manual com trecho da Actina: {seq_teste}")

if 'blacklist_kmers' in globals():
    # Testa normal
    if seq_teste in blacklist_kmers:
        print("   [ESTRANHO] O k-mer EXISTE no banco. O scanner deveria ter achado.")
    else:
        print("   [FALHA] O k-mer NÃO foi encontrado no banco.")
        
    # Testa minúsculo (para confirmar hipótese 2)
    if seq_teste.lower() in blacklist_kmers:
        print("   [DIAGNÓSTICO FINAL] O k-mer foi achado em MINÚSCULO! Problema de .upper()/.lower().")

print("\n--- FIM DO DIAGNÓSTICO ---")

--- INICIANDO DIAGNÓSTICO DO SISTEMA ---
1. Tamanho do Banco de Citros na memória: 29741838 k-mers
   [OK] O banco parece ter dados.
2. Amostra de um k-mer do Citros: 'GAGGTCTGGAATGCGAATGAG'
   [OK] O banco está em MAIÚSCULAS.

3. Teste Manual com trecho da Actina: ATGGCTGGTGGCCGCAATACC
   [FALHA] O k-mer NÃO foi encontrado no banco.

--- FIM DO DIAGNÓSTICO ---


In [40]:
# VALIDAÇÃO FINAL: O Teste do Espião (Positive Control Absoluto)
from Bio import SeqIO
import random

print("--- INICIANDO VALIDAÇÃO 'ESPIÃO' ---")
print("Objetivo: Testar um gene original do Citros contra o próprio banco do Citros.")
print("Resultado Esperado: O mapa deve ser COMPLETAMENTE VERMELHO (X).")

# 1. Pescar um gene aleatório do arquivo do Citros (Hospedeiro)
arquivo_citrus = c_sin_CDS_file
id_espiao = ""
seq_espiao = ""

try:
    # Lê os primeiros 100 genes e pega um aleatório para não viciar
    candidatos = []
    limit = 0
    for record in SeqIO.parse(arquivo_citrus, "fasta"):
        # Pega genes com tamanho razoável (>500pb) para o mapa ficar bonito
        if len(record.seq) > 500:
            candidatos.append(record)
        limit += 1
        if limit > 200: break
    
    # Seleciona o "Espião"
    gene_escolhido = candidatos[50] # Pega o 50º gene da lista
    id_espiao = gene_escolhido.id
    seq_espiao = str(gene_escolhido.seq).upper()
    
    print(f"\n[GENE ESPIÃO SELECIONADO]")
    print(f"ID Original (Citros): {id_espiao}")
    print(f"Descrição: {gene_escolhido.description}")
    print(f"Tamanho: {len(seq_espiao)} pb")

    # 2. Rodar o Scanner contra a Blacklist (que já está na memória)
    if 'blacklist_kmers' not in globals():
        print("[ERRO] Blacklist não carregada. Rode a Célula 7.")
    else:
        print("\n--- RODANDO SCANNER... ---")
        conflitos = 0
        mapa = []
        
        for i in range(len(seq_espiao) - 21 + 1):
            kmer = seq_espiao[i : i+21]
            if kmer in blacklist_kmers:
                conflitos += 1
                mapa.append("X")
            else:
                mapa.append(".") # Isso não deveria acontecer num gene idêntico
        
        # 3. Relatório
        print(f"K-mers analisados: {len(seq_espiao) - 20}")
        print(f"Conflitos Encontrados: {conflitos}")
        
        percentual_conflito = (conflitos / (len(seq_espiao) - 20)) * 100
        print(f"Similaridade Detectada: {percentual_conflito:.1f}%")
        
        print("\n[MAPA VISUAL - Espera-se tudo 'X']")
        str_mapa = "".join(mapa)
        print(f"{str_mapa[:100]}... (primeiros 100nt)")
        
        if percentual_conflito > 95:
            print("\n>>> VEREDITO: O SCRIPT FUNCIONA PERFEITAMENTE. <<<")
            print("O sistema detectou que esse gene é do hospedeiro.")
            print("Conclusão: O resultado 'seguro' da Actina do fungo é real (divergência evolutiva).")
        else:
            print("\n>>> VEREDITO: FALHA NO SISTEMA. <<<")
            print("O script não reconheceu um gene do próprio banco.")

except Exception as e:
    print(f"Erro ao ler arquivo do Citros: {e}")

--- INICIANDO VALIDAÇÃO 'ESPIÃO' ---
Objetivo: Testar um gene original do Citros contra o próprio banco do Citros.
Resultado Esperado: O mapa deve ser COMPLETAMENTE VERMELHO (X).

[GENE ESPIÃO SELECIONADO]
ID Original (Citros): lcl|NC_068556.1_cds_XP_006483619.2_56
Descrição: lcl|NC_068556.1_cds_XP_006483619.2_56 [gene=LOC102617211] [db_xref=GeneID:102617211] [protein=uncharacterized protein LOC102617211] [protein_id=XP_006483619.2] [location=complement(join(206606..207522,208030..208123,208480..208674,208781..208853,209366..209523,210004..210133,210221..210296,210377..210434,210530..210628,210705..210833,211424..211545,211739..211907,212228..212334,213124..213250,213338..213427,213559..213645,213939..214166,214248..214322))] [gbkey=CDS]
Tamanho: 2934 pb

--- RODANDO SCANNER... ---
K-mers analisados: 2914
Conflitos Encontrados: 2914
Similaridade Detectada: 100.0%

[MAPA VISUAL - Espera-se tudo 'X']
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

In [41]:
# --- ESCOLHA DO ALVO: Tubulina do Fungo ---
id_gene_alvo = "lcl|SDAQ01000015.1_cds_KAI3555828.1_3881"  
varredura_especificidade(id_gene_alvo, c_abs_genes, blacklist_kmers)

--- Analisando Especificidade do Gene: lcl|SDAQ01000015.1_cds_KAI3555828.1_3881 ---

Resumo da Análise:
1. Tamanho do Gene: 1353 pb
2. Conflitos (Off-targets): 0 k-mers perigosos encontrados.
3. Índice de Segurança Global: 100.0%

[MAPA VISUAL SIMPLIFICADO]
0000: ....................................................................................................
0100: ....................................................................................................
0200: ....................................................................................................
0300: ....................................................................................................
0400: ....................................................................................................
0500: ....................................................................................................
0600: ..............................................................................................

In [42]:
# O Teste Comparativo (Fungo vs Espião de Planta)
from Bio import SeqIO

# --- 1. CONFIGURAÇÃO ---
id_tubulin_fungo = "lcl|SDAQ01000015.1_cds_KAI3555828.1_3881"
arquivo_citrus = c_sin_CDS_file

# Verifica se o banco está na memória
if 'blacklist_kmers' not in globals():
    print("[AVISO] Carregando banco de Citrus (pode demorar uns segundos)...")
    blacklist_kmers = set()
    try:
        # Tenta carregar (lógica simplificada para garantir execução)
        handle = open(arquivo_citrus, "r")
        for record in SeqIO.parse(handle, "fasta"):
            seq = str(record.seq).upper()
            for i in range(len(seq) - 21 + 1):
                blacklist_kmers.add(seq[i:i+21])
        handle.close()
        print(f"[OK] Banco carregado com {len(blacklist_kmers)} k-mers.")
    except Exception as e:
        print(f"[ERRO] Falha ao carregar Citrus: {e}")

# Função Auxiliar de Scanner
def rodar_scanner(seq, titulo):
    conflitos = 0
    mapa = []
    seq = seq.upper()
    for i in range(len(seq) - 21 + 1):
        if seq[i:i+21] in blacklist_kmers:
            conflitos += 1
            mapa.append("X")
        else:
            mapa.append(".")
    
    percentual = (conflitos / (len(seq)-20)) * 100
    print(f"\n>>> ANÁLISE: {titulo}")
    print(f"    Tamanho: {len(seq)} pb")
    print(f"    Similaridade (Risco): {percentual:.1f}%")
    print(f"    Mapa Visual: {''.join(mapa)[:60]}...")
    return percentual

# --- 2. EXECUÇÃO DO TESTE DO FUNGO ---
if id_tubulin_fungo in c_abs_genes:
    seq_fungo = str(c_abs_genes[id_tubulin_fungo].seq)
    score_fungo = rodar_scanner(seq_fungo, "TUBULINA DO FUNGO (Alpha-B)")
else:
    print(f"[ERRO] ID {id_tubulin_fungo} não encontrado no genoma do fungo.")

# --- 3. EXECUÇÃO DO "ESPIÃO" (TUBULINA DA PLANTA) ---
# Vamos caçar uma tubulina real dentro do arquivo do Citrus
print("\n[BUSCANDO TUBULINA 'ESPIÃO' NO GENOMA DE CITRUS...]")
seq_espiao = None
id_espiao = None

try:
    # Varredura rápida no arquivo de Citrus para achar uma Tubulina
    for record in SeqIO.parse(arquivo_citrus, "fasta"):
        if "tubulin alpha" in record.description.lower():
            seq_espiao = str(record.seq)
            id_espiao = record.id
            print(f"    Encontrada! {record.description[:60]}...")
            break
            
    if seq_espiao:
        score_espiao = rodar_scanner(seq_espiao, "CONTROLE POSITIVO: TUBULINA DO CITROS")
        
        # --- VEREDITO FINAL ---
        print("\n" + "="*40)
        print("RESULTADO DA VALIDAÇÃO DO SOFTWARE")
        print("="*40)
        
        if score_espiao > 90:
            print("✅ SISTEMA APROVADO: O software detectou corretamente o gene da planta.")
            if score_fungo < 10:
                print("✅ BIOLOGIA CONFIRMADA: A tubulina do fungo é segura (Wobble effect).")
                print("   Conclusão: O alvo é ESPECÍFICO.")
            else:
                print("⚠️ ALERTA: A tubulina do fungo tem risco real de silenciamento!")
        else:
            print("❌ SISTEMA REPROVADO: O software falhou em reconhecer a própria planta.")
            
    else:
        print("[ERRO] Não encontrei nenhuma tubulina no arquivo de Citrus para comparar.")

except Exception as e:
    print(f"Erro na busca do espião: {e}")


>>> ANÁLISE: TUBULINA DO FUNGO (Alpha-B)
    Tamanho: 1353 pb
    Similaridade (Risco): 0.0%
    Mapa Visual: ...............................................................

[BUSCANDO TUBULINA 'ESPIÃO' NO GENOMA DE CITRUS...]
    Encontrada! lcl|NC_068557.1_cds_XP_006488930.1_3248 [gene=LOC102620032] ...

>>> ANÁLISE: CONTROLE POSITIVO: TUBULINA DO CITROS
    Tamanho: 1350 pb
    Similaridade (Risco): 100.0%
    Mapa Visual: XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...

RESULTADO DA VALIDAÇÃO DO SOFTWARE
✅ SISTEMA APROVADO: O software detectou corretamente o gene da planta.
✅ BIOLOGIA CONFIRMADA: A tubulina do fungo é segura (Wobble effect).
   Conclusão: O alvo é ESPECÍFICO.


Segundo Gemini, não encontrou a actina e a tubulina devido à degeneração do código genético (os genes diferem em algumas bases mas a proteina produzida é a mesma).